In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.profiler import profile, record_function, ProfilerActivity
from numba import cuda
import numpy as np
import math

In [ ]:
class TransformerBase(nn.Module):
    def __init__(self, vocab_size=1000, d_model=512, n_heads=4, d_ff=512 * 4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.n_heads = n_heads
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        raise NotImplementedError()

In [ ]:
class CpuPipeline(TransformerBase):
    def forward(self, x):
        # x: [N]
        # ['heelo', 'im' , 'a', 'dog']
        with record_function("1_embedding"):
            x = self.embedding(x) # [N, D]

        B, N, D = x.shape

        q = self.q_proj(x)  #  [x(N, D) . Wq(D, k)] -> [N, k]
        k = self.k_proj(x)  # [N, k] -> kT []
        v = self.v_proj(x)  # [N, k]

        with record_function("2_attention_total"):

            with record_function("2a_qk_matmul"):
                attn_scores = torch.matmul(q, k.transpose(-2, -1)) / D ** 0.5

            # x: N N

            with record_function("2b_softmax"):
                attn_weights = torch.softmax(attn_scores, dim=-1)

            with record_function("2c_value_weighted_sum"):
                attn_out = torch.matmul(attn_weights, v)

        x = self.norm1(x + attn_out)

        with record_function("3_ffn"):
            ffn_out = self.ffn(x)
            x = self.norm2(x + ffn_out)

        return x

In [ ]:
@cuda.jit
def _matmul(A, B, C):
    i, j = cuda.grid(2)
    if i < C.shape[0] and j < C.shape[1]:
        tmp = 0.
        for k in range(A.shape[1]):
            tmp += A[i, k] * B[k, j]
        C[i, j] = tmp

In [ ]:
@cuda.jit
def _softmax(input, out):
    j = cuda.grid(1)
    if j < input.shape[1]:
        tmp = 0.
        for i in range(input.shape[0]):
            tmp += math.exp(input[i, j])
        for i in range(input.shape[0]):
            out[i, j] = math.exp(input[i, j]) / tmp

In [ ]:
class GpuV1(TransformerBase):
    def forward(self, x):
        # x: [N]
        # ['heelo', 'im' , 'a', 'dog']
        with record_function("1_embedding"):
            x = self.embedding(x) # [N, D]

        B, N, D = x.shape

        q = self.q_proj(x).squeeze(0).cuda()  # [x(N, D) . Wq(D, k)] -> [N, k]
        k = self.k_proj(x).squeeze(0).cuda()  # [N, k] -> kT []
        v = self.v_proj(x).squeeze(0).cuda()  # [N, k]

        attn_scores = torch.empty((N, N), device=torch.device('cuda'))
        attn_weights = torch.empty((N, N), device=torch.device('cuda'))
        attn_out = torch.empty((N, D // self.n_heads), device=torch.device('cuda'))
        with record_function("2_attention_total"):
            with record_function("2a_qk_matmul"):
                kT = k.transpose(-2, -1)
                threadsperblock = (16, 16)
                blockspergrid_x = int(np.ceil(q.shape[0] / threadsperblock[0]))
                blockspergrid_y = int(np.ceil(kT.shape[1] / threadsperblock[1]))
                blockspergrid = (blockspergrid_x, blockspergrid_y)

                _matmul[blockspergrid, threadsperblock](q, kT, attn_scores)
                # x [N N]

                attn_scores = attn_scores / D ** 0.5

            with record_function("2b_softmax"):
                tpb = 256
                bpg = int(np.ceil(attn_scores.shape[1] / tpb))
                _softmax[bpg, tpb](attn_scores, attn_weights)

            with record_function("2c_value_weighted_sum"):
                _matmul(attn_weights, v, attn_out)

        attn_out = attn_out.to(torch.device('cpu'))

        x = self.norm1(x + attn_out)

        with record_function("3_ffn"):
            ffn_out = self.ffn(x)
            x = self.norm2(x + ffn_out)

        return x


In [ ]:
class GpuV2(TransformerBase):
    pass

In [ ]:
class GpuV3(TransformerBase):
    pass

In [ ]:
N = 2 ** 15
B = 1
inputs = torch.randint(0, 1000, (B, N)) # [B, N]

In [ ]:
model = CpuPipeline().eval()
with profile(
    activities=[ProfilerActivity.CPU],
    record_shapes=True,
    profile_memory=True
) as prof:
    with torch.no_grad():
        with record_function("total_inference_pipeline"):
            _ = model(inputs)
print(prof.key_averages().table(sort_by="cpu_time_total"))


----------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                        Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  
----------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
    total_inference_pipeline         1.28%     375.663ms       100.00%       29.453s       29.453s      62.00 MB      -8.44 GB             1  
           2_attention_total         0.00%     197.729us        91.21%       26.865s       26.865s       8.06 GB           0 B             1  
                aten::matmul         0.00%     357.948us        73.15%       21.546s        4.309s       4.25 GB           0 B             5  
                   aten::bmm        70.90%       20.882s        70.90%       20.882s       10.441s       4.06 GB       4.06 GB             2  

In [ ]:
model = GpuV1().eval()
with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=True,
    profile_memory=True
) as prof:
    with torch.no_grad():
        with record_function("total_inference_pipeline"):
            _ = model(inputs)
print(prof.key_averages().table(sort_by="cpu_time_total"))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                               total_inference_pipeline         0.30%      43.934ms       100.00%       14.434s       14.434s       0.000us         0.00%     444.810ms     444.810ms           0 B    -512.00 MB           0 B      -8.25 G

In [ ]:
n = 100
A = np.arange(n ** 2).reshape([n, n])
B = np.ones((n, n))
C = np.zeros((n, n))
A = cuda.to_device(A)
B = cuda.to_device(B)
C = cuda.to_device(C)
threadsperblock = (16, 16)
blockspergrid_x = int(np.ceil(A.shape[0] / threadsperblock[0]))
blockspergrid_y = int(np.ceil(B.shape[1] / threadsperblock[1]))
blockspergrid = (blockspergrid_x, blockspergrid_y)
_matmul[blockspergrid, threadsperblock](A, B, C)
print(C.copy_to_host())